In [ ]:
import pandas as pd

def create_data_dict_df(data_dict_file_name:str, data_dict_append_file_name) -> pd.DataFrame:
    """
    Creates and returns the data dictionary DataFrame.

    Args:
        data_dict_file_name (str): The file name of the main data dictionary CSV.
        data_dict_append_file_name (str): The file name of the additional data dictionary CSV to append.

    Returns:
        pd.DataFrame: The combined data dictionary DataFrame.
    """
    # Read the main data dictionary CSV
    data_dict_df = pd.read_csv(data_dict_file_name, na_filter=False, dtype=str)
    
    # Read the additional data dictionary CSV and append it to the main DataFrame
    data_dict_append_df = pd.read_csv(data_dict_append_file_name, na_filter=False, dtype=str)
    data_dict_df = pd.concat([data_dict_df, data_dict_append_df], ignore_index=True)
    
    # Add a sequence number to the data dictionary
    data_dict_df.insert(0, 'sequence', range(1, len(data_dict_df) + 1))
    
    # Set sequence dtype to int
    data_dict_df['sequence'] = data_dict_df['sequence'].astype(int)
    
    # Select and rename columns
    data_dict_df = data_dict_df[['sequence', 'Variable / Field Name', 'Form Name', 'Field Type', 'Field Label']]
    data_dict_df = data_dict_df.rename(columns={'Variable / Field Name': 'field_name', 'Form Name': 'form_name', 
                                                'Field Type': 'field_type', 'Field Label': 'field_label'})
    
    # Add CreatedOn and FileName columns
    data_dict_df['CreatedOn'] = pd.to_datetime('today').date()
    data_dict_df['FileName'] = data_dict_file_name.split('/')[-1]
    
    # Set the data types for all fields in the data_dict_df
    data_dict_df = data_dict_df.astype({'sequence': int, 'field_name': str, 'form_name': str, 'field_type': str, 
                                        'field_label': str, 'CreatedOn': 'datetime64[ns]', 'FileName': str})
    
    # Select and rename columns
    data_dict_df = data_dict_df[['sequence', 'form_name', 'field_name', 'field_label', 'field_type', 'FileName', 'CreatedOn']]
    data_dict_df = data_dict_df.rename(columns={'sequence': 'SequenceId', 'field_name': 'FieldName', 'form_name': 'FormName', 
                                                'field_type': 'FieldType', 'field_label': 'FieldLabel'})
    
    return data_dict_df

#### Clinical Abstraction

In [ ]:
#!/usr/bin/env python
import requests
import pandas as pd
data = {
    'token': '36526C109F7F5B0B454921FBF2BF0821',
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    'csvDelimiter': '',
    'rawOrLabel': 'raw',
    'rawOrLabelHeaders': 'raw',
    'exportCheckboxLabel': 'true', # ignored if 'type': 'eav'
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}
r = requests.post('https://champs.emory.edu/redcap/api/',data=data)
print(f'HTTP Status: {r.status_code}')
# print(r.json())
# read in the response into a pandas dataframe


df = pd.DataFrame.from_dict(r.json())
# print(df.head())
data_dict_df = pd.read_csv('data/AdultHIVClinicalAbstr_DataDict_2024-10-30.csv')
# add a sequence number to the data dictionary
data_dict_df.insert(0, 'sequence', range(1, len(data_dict_df) + 1))
# set sequence dtype to int
data_dict_df['sequence'] = data_dict_df['sequence'].astype(int)
data_dict_df = data_dict_df[['sequence', 'Variable / Field Name', 'Form Name', 'Field Type', 'Field Label']]
data_dict_df = data_dict_df.rename(columns={'Variable / Field Name': 'field_name', 'Form Name': 'form_name', 
                                            'Field Type': 'field_type', 'Field Label': 'field_label'})

# in field_label column replace all <br> with space and any "," with ";"
data_dict_df['field_label'] = data_dict_df['field_label'].str.replace('<br>', ' ', regex=False).str.replace(',', ';', regex=False).replace('\n', ' ', regex=True)
# df['value'] = df['value'].str.replace('<br>', ' ', regex=False).str.replace(',', ';', regex=False).replace('\n', ' ', regex=True)
df = df.merge(data_dict_df, on='field_name', how='left')

# create df with unique record, field_name, and value columns where value == 'am1_country'
# This will be used to create a new column called 'SiteId' in the original df 
# The 'SiteId' column will contain the value of the 'value' column from the df_am1_country df for each record
df_am1_country = df[(df['field_name'] == 'am1_country') & (df['value'] != '')][['record', 'value']]
df_am1_country = df_am1_country.rename(columns={'value': 'SiteId'})
print(df_am1_country.head())

# merge with original df
df = df.merge(df_am1_country, on=['record'], how='left')
print(df.head())

df.to_csv('data/redcap_hiv.csv', index=False)

#### Clinical Abstraction extract and load

In [40]:
# If the data/redcap_hiv.csv is not empty the insert into staging table
import pandas as pd
import sqlalchemy as sa
from ci_utils import connect_db
CSV_FILE_NAME = 'data/redcap_hiv.csv'
STG_TABLE_NAME = 'HIVClinicalAbstract_stg'
DATA_DICT_FILE_NAME = 'data/AdultHIVClinicalAbstr_DataDict_2024-10-30.csv'
DATA_DICT_APPEND_FILE_NAME = 'data/AdultHIVClinicalAbstr_DataDict_append.csv'
df = pd.read_csv(CSV_FILE_NAME, dtype=str)
if not df.empty:
    engine = connect_db.conn_qa_rpt
    with engine.connect() as conn:
        data_dict_df = create_data_dict_df(DATA_DICT_FILE_NAME, DATA_DICT_APPEND_FILE_NAME)
        
        truncate_stg = sa.text(f"TRUNCATE TABLE stg.{STG_TABLE_NAME}")
        conn.execute(truncate_stg)

        df = df.rename(columns={'record': 'ChampsId', 'field_name': 'FieldName', 'value': 'FieldValue'})
        df = df[['SiteId', 'ChampsId', 'FieldName', 'FieldValue']]
        
        # Set column data types
        df = df.astype({'SiteId': str, 'ChampsId': str, 'FieldName': str, 'FieldValue': str}) 
        df['SiteId'] = df['SiteId'].astype(str)
        # print(df.head())
        df.to_sql(STG_TABLE_NAME,schema='stg', con=conn, if_exists='append', index=False)
        data_dict_df.to_sql('HIVDataDictClinicalAbstr', conn, schema='stg', if_exists='replace', index=False) if not data_dict_df.empty else None
        print(f"Inserted {len(df)} rows into {STG_TABLE_NAME}")

        conn.commit()

Inserted 958 rows into HIVClinicalAbstract_stg


### create Consent tracking record for champs_id
Only create if champs_id is not null in 1.1 <br>
https://champs.emory.edu/redcap/redcap_v13.8.1/DataEntry/record_status_dashboard.php?pid=441
EC8BCA8BC37AAAC6A85A5B163EDD1B07

Package 1.1 ETL

In [2]:
import requests
import pandas as pd
import duckdb
# from ci_utils import connect_db

# pid=441 project 1.1 
data = {
    'token': 'EC8BCA8BC37AAAC6A85A5B163EDD1B07',
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    'csvDelimiter': '',
    'rawOrLabel': 'raw',
    'rawOrLabelHeaders': 'raw',
    'exportCheckboxLabel': 'true', # ignored if 'type': 'eav'
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'false',
    'returnFormat': 'json'
}
r = requests.post('https://champs.emory.edu/redcap/api/',data=data)
print(f'HTTP Status: {r.status_code}')
df = pd.DataFrame.from_dict(r.json())

# use duckdb for transformation
if r.status_code != 200:
    print('Error fetching data from REDCap')
    exit()
if df.empty:
    print('No data fetched from REDCap for Project 1.1')
    exit()   
df = df.astype(str)
conn = duckdb.connect()
conn.execute(" CREATE TABLE redcap_hiv_project1_1 AS SELECT * FROM df ")
conn.execute(" COPY redcap_hiv_project1_1 TO 'data/redcap_hiv_project1_1.parquet' (FORMAT 'parquet') ")
conn.execute("""CREATE or REPLACE table redcap_hiv_project1_1 as
                select record,field_name,value 
                FROM read_parquet('data/redcap_hiv_project1_1.parquet')
             """)

df_site = conn.execute(""" WITH site_cte as (
                                SELECT record,
                                MAX(CASE WHEN field_name = 'champs_id' THEN value END) AS champs_id,
                                MAX(CASE WHEN field_name = 'site_id' THEN value END) AS site_id
                                ,max(case when field_name like 'catchment_id%' THEN value END) AS catchment_id
                                FROM redcap_hiv_project1_1
                                WHERE (field_name in ('champs_id', 'site_id' ) or field_name like 'catchment_id%')
                                GROUP BY record 
                            )
                            SELECT p.*, site_cte.site_id, site_cte.catchment_id, site_cte.champs_id 
                            from redcap_hiv_project1_1 p
                            join site_cte on p.record = site_cte.record 
                        """).fetch_df()
print(df_site.count())


# df.to_csv('data/redcap_hiv_project1_1.csv', index=False)
df_site.to_csv('data/redcap_hiv_project1_1_site.csv', index=False)

HTTP Status: 200
record          452
field_name      452
value           452
site_id         452
catchment_id    340
champs_id       310
dtype: int64


####Project 1.1 insert into staging table
Create table stg.HIVNotificationConsent_stg (
    Id varchar(100) primary key default newid(),
    SiteId varchar(10),
    ReportId varchar(100),
    FieldName varchar(255),
    FieldValue varchar(max),
    CreatedOn datetime default getdate()
);

In [ ]:
# insert into staging table
####Project 1.1 insert into staging table
# Create table stg.HIVNotificationConsent_stg (
#     Id varchar(100) primary key default newid(),
#     SiteId varchar(10),
#     ReportId varchar(100),
#     FieldName varchar(255),
#     FieldValue varchar(max),
#     CreatedOn datetime default getdate()
# );

import pandas as pd
import sqlalchemy as sa
from ci_utils import connect_db
import duckdb
CSV_FILE_NAME = 'data/redcap_hiv_project1_1_site.csv'
STG_TABLE_NAME = 'HIVProject1_1_stg'
DATA_DICT_TABLE_NAME = 'HIVDataDictProj1_1'
DATA_DICT_FILE_NAME = 'data/AdultHIVProject1_1_DataDict_2024-12-11.csv'
DATA_DICT_APPEND_FILE_NAME = 'data/AdultHIVProject1_1_DataDict_append.csv'


df = pd.read_csv(CSV_FILE_NAME, dtype=str, na_filter=False)
if not df.empty:
    engine = connect_db.conn_qa
    with engine.connect() as conn:
        data_dict_df = create_data_dict_df(DATA_DICT_FILE_NAME, DATA_DICT_APPEND_FILE_NAME)
        truncate_stg = sa.text(f"TRUNCATE TABLE stg.{STG_TABLE_NAME}")
        conn.execute(truncate_stg)
        df = df.rename(columns={'record': 'ReportId','site_id':'SiteId', 'catchment_id': 'CatchmentId'  ,'champs_id': 'ChampsId' , 
                                'field_name': 'FieldName', 'value': 'FieldValue'})
        df = df.astype({'SiteId': str,'CatchmentId': str, 'ReportId': str, 'ChampsId':str ,'FieldName': str, 'FieldValue': str}) 
        df = df[['SiteId','CatchmentId', 'ReportId', 'ChampsId', 'FieldName', 'FieldValue']]
        # df.to_csv('data/redcap_hiv_project1_1_db_temp.csv', index=False)
        # print("df data type",df.dtypes)
        # TODO: add a warning that catchment_id are missing in the data
        df.to_sql(STG_TABLE_NAME, conn, schema='stg', if_exists='append', index=False)
        data_dict_df.to_sql(DATA_DICT_TABLE_NAME, conn, schema='stg', if_exists='replace', index=False) if not data_dict_df.empty else None
        conn.commit()

In [28]:
# Insert consents in consentTtracking table
# adult_hiv_study
import pandas as pd
import sqlalchemy as sa
from ci_utils import connect_db
import os
CONSENTTRACK_VIEW_NAME = 'vw_HIVConsentTracking'
SCHEMA_NAME = 'stg'

engine = connect_db.conn_qa

MERGE_SQL = sa.text (f""" MERGE into [dbo].[ConsentTracking] as Target
    using {SCHEMA_NAME}.{CONSENTTRACK_VIEW_NAME} as Source
    on (
        Target.ReportId = Source.ReportId 
        and Target.CatchmentId = Source.CatchmentId
        and  Target.ChampsId = Source.ChampsId 
        and Target.SiteId = Source.SiteId 
        and Target.[FileName] = Source.[FileName]
        and target.Filename = 'adult_hiv_study' and Target.Active=1 ) 
    when matched then
        update set Target.CatchmentId = Source.CatchmentId,
                Target.ConsentGranted = Source.ConsentGranted,
                Target.ConsentType = Source.ConsentType,
                Target.ConsentDate = Source.ConsentDate,
                Target.ModifiedOn = Source.CreatedOn,
                Target.UploadedOn = Source.CreatedOn
    when not matched then
        insert (Id, ConsentTrackingSiteId, ReportId, CatchmentId, ChampsId, ConsentGranted, ConsentType, ConsentDate, SiteId, [FileName], CreatedOn, ModifiedOn, Uploadedon, Active)
        values (Source.Id, Source.ConsentTrackingSiteId, Source.ReportId, Source.CatchmentId, Source.ChampsId, Source.ConsentGranted, 
        Source.ConsentType, Source.ConsentDate, Source.SiteId, Source.FileName, Source.CreatedOn, Source.CreatedOn, Source.CreatedOn, Source.Active)
    ; 
""")
# ConsentTracking set duplicate records to inactive
DEACTIVATE_DUPS_SQL= sa.text("""
        update [ConsentTracking] set Active = 0 
        where FileName = 'adult_hiv_study' 
        and Id in 
            ( select Id from 
                (select Id, SiteId, ChampsId,FileName, CatchmentId, createdon, ModifiedOn, UploadedOn, Active, 
                ROW_NUMBER() over (partition by ChampsId order by CreatedOn desc) as RowNumber_dup 
                from ConsentTracking  
                where FileName = 'adult_hiv_study' 
                and Active = 1) as dup 
                where RowNumber_dup > 1 
                ) ;
    """
)

# deactivate any record not found in source CONSENTTRACK_VIEW_NAME 
# on the basis of ChampsId, SiteId, FileName, Active = 1
#  set active = 2
DEACTIVATE_DELETED_SQL = sa.text(f""" 
                                update [ConsentTracking] set Active = 2 , ModifiedOn = getdate()
                                where FileName = 'adult_hiv_study'
                                and Active = 1
                                and not exists (select 1 from {SCHEMA_NAME}.{CONSENTTRACK_VIEW_NAME} source 
                                    where source.ChampsId = ConsentTracking.ChampsId
                                    and source.SiteId = ConsentTracking.SiteId
                                    and source.[FileName] = ConsentTracking.[FileName]
                                    and source.Active = 1
                                ) ;
                                """)
                                

with engine.connect() as conn:
    result = conn.execute(MERGE_SQL)
    result_deactivate= conn.execute(DEACTIVATE_DUPS_SQL)
    result_deactivate_deleted = conn.execute(DEACTIVATE_DELETED_SQL)
    conn.commit()
    print(f"Merged {result.rowcount} rows into {CONSENTTRACK_VIEW_NAME}, Deactivated {result_deactivate.rowcount} duplicate rows, \
          Deactivated {result_deactivate_deleted.rowcount} rows not found in source")


Merged 3 rows into vw_HIVConsentTracking, Deactivated 0 duplicate rows,           Deactivated 0 rows not found in source


#### TODO: Add 1.1 record in DeathNotification Table
This maybe required for the MITS report to be called

```sql
insert into DeathNotification (Id, ReportId,CatchmentId, DateOfDeathNotification, SiteId, DeathNotificationSiteId, FileName, isCPLReport)
-- select newid(), ChampsId, DateOfDeathNotification, SiteGuid, IsCPLReport
SELECT newid() Id , -- FormName, 
    ReportId, CatchmentId,  
    -- ChampsId, FieldName, 
    FieldValue DateOfDeathNotification, 
    Site.Id SiteId,Site.SiteId DeathNotificationSiteId,
     'adult_hiv_study' FileName
        FROM stg.vw_HIVProject1_1 
        join Site on Site.SiteId = vw_HIVProject1_1.SiteId
        WHERE FormName like 'initial_death_notification'
         AND FieldName IN ('report_death_dt' )
        and ChampsId = 'DWHV00001'
;
```

In [ ]:
# use view stg.vw_HIVDeathNotificationto upsert into DeathNotification table
#
import pandas as pd
import sqlalchemy as sa
from ci_utils import connect_db
HIV_DEATH_NOTIFICATION_VIEW = 'vw_HIVDeathNotification'
SCHEMA_NAME = 'stg'

engine = connect_db.conn_qa

MERGE_SQL = sa.text(f"""
    MERGE into [dbo].[DeathNotification] as Target
    using {SCHEMA_NAME}.{HIV_DEATH_NOTIFICATION_VIEW} as Source
                    on (
                        Target.ReportId = Source.ReportId 
                        and Target.CatchmentId = Source.CatchmentId
                        and Target.[FileName] = Source.[FileName]
                        and Target.FileName = 'adult_hiv_study'
                        and Target.Active=1 )
    when matched AND (Target.DateOfDeathNotification != Source.DateOfDeathNotification ) then
        update set Target.DateOfDeathNotification = Source.DateOfDeathNotification,
                    Target.ModifiedOn = Source.ModifiedOn,
                    Target.UploadedOn = Source.UploadedOn
    when not matched then
        insert (Id, DeathNotificationSiteId, ReportId, CatchmentId, DateOfDeathNotification, 
                    SiteId, [FileName], CreatedOn, ModifiedOn, UploadedOn, Active)
        values (Source.Id, Source.DeathNotificationSiteId, Source.ReportId, Source.CatchmentId, Source.DateOfDeathNotification, 
                    Source.SiteId, Source.FileName, Source.CreatedOn, Source.ModifiedOn, Source.UploadedOn, Source.Active)  
                    ;
                    """)

#  Deactivate duplicate records based on SiteId, ReportId, CatchmentId
#  set duplicate records to inactive (Active = 0) 
DEACTIVE_DUPS_SQL= sa.text(""" 
        update DeathNotification set Active = 0 
        where FileName = 'adult_hiv_study' 
        and Id in 
        (   SELECT Id from
                (  SELECT Id, ReportId CatchmentId
                    , ROW_NUMBER() over (partition by SiteId, ReportId, CatchmentId order by CreatedOn desc) as RowNumber_dup 
                    from DeathNotification  
                    where FileName = 'adult_hiv_study' 
                    and Active = 1) dup
            where RowNumber_dup > 1
                ) ;               
        """)

#  DEactivate any records not found in source stg.vw_HIVDeathNotification
# based on the FileName = 'adult_hiv_study' and SiteId, ReportId, CatchmentId
DEACTIVATE_DELETED_SQL = sa.text(f"""
        update DeathNotification set Active = 2 , ModifiedOn = getdate() -- set to 2 to indicate deleted source records
        where FileName = 'adult_hiv_study' 
        and Active = 1
        and not exists 
            ( select 1 from {SCHEMA_NAME}.{HIV_DEATH_NOTIFICATION_VIEW} as Source 
                where Source.ReportId = DeathNotification.ReportId 
                and Source.CatchmentId = DeathNotification.CatchmentId
                and Source.FileName = DeathNotification.FileName
                and Source.SiteId = DeathNotification.SiteId
            );
        """)

with engine.connect() as conn:
    result = conn.execute(MERGE_SQL)
    result_deactivate= conn.execute(DEACTIVE_DUPS_SQL)
    result_deactivate_deleted = conn.execute(DEACTIVATE_DELETED_SQL)
    conn.commit()
    print(f"Merged  {result.rowcount} rows into DeathNotification table and Deactivated {result_deactivate.rowcount} duplicate rows. \
        Deactivated {result_deactivate_deleted.rowcount} rows not found in source")


Merged  0 rows into DeathNotification table and Deactivated 0 duplicate rows.         Deactivated 0 rows not found in source


#### Generate Tableau report Adult-HIV-Clinical-Abstraction.pdf


In [ ]:
# create tableau connection and generate a pdf of the workbook 
# workbook name is Adult-HIV-Clinical-Abstraction
import tableauserverclient as TSC
from include.ci_utils import tab_server, tab_user, tab_pwd, tab_site
import os
import PyPDF2 as pypdf2
TAB_SERVER_DEV = 'https://tableau.emory.edu/'
tableau_auth = TSC.TableauAuth(username=tab_user, password=tab_pwd, site=tab_site)
server = TSC.Server(server_address=TAB_SERVER_DEV, use_server_version=True)

with server.auth.sign_in(tableau_auth):
    print(f"Site Id is : {server.site_id}")
    all_workbooks_items, pagination_info = server.workbooks.get()
    for workbook in all_workbooks_items:
        if workbook.name.upper().startswith ('Adult-HIV-Project1_1'.upper()): 
                                            #('Adult-HIV-Clinical-Abstraction'.upper()):
            print(workbook.name, workbook.id)

            # generate a pdf of the workbook id 7a3e971e-6404-4ad5-8724-c967a9bd6395
            workbook = server.workbooks.get_by_id(workbook.id)
            server.workbooks.populate_views(workbook)
            
            # Iterate through each view in the workbook
            for view in workbook.views:
                # print the workbook name and view name, id
                print(workbook.name, view.name, view.id)
                # Set PDF options
                pdf_options = TSC.PDFRequestOptions(page_type=TSC.PDFRequestOptions.PageType.A3, maxage=1)
                pdf_options.vf('ChampsId', 'DWHV00001') 

                # Generate PDF for the current view
                server.views.populate_pdf(view, req_options=pdf_options)

                # Save the PDF for the current view with workbook name and view name
                with open(f'data/{workbook.name}_{view.name}.pdf', 'wb') as f:
                    f.write(view.pdf) 

            # # merge all pdfs into one pdf
            # pdf_merger = pypdf2.PdfMerger()
            # merged_pdf_file = f'data/{workbook.name}.pdf'
            # for pdf_file in os.listdir('data'):
            #     if pdf_file.endswith('.pdf') and pdf_file.startswith(f'{workbook.name}'):
            #         # create a merged pdf named Adult-HIV-Clinical-Abstraction.pdf
            #         with open(os.path.join('data', pdf_file), 'rb') as f:
            #             pdf_merger.append(f)
            # with open(merged_pdf_file, 'wb') as f:
            #     pdf_merger.write(f)


Site Id is : 606868a1-e41d-4330-9a5f-57959f6c8272
Adult-HIV-Project1_1 5da87775-4508-4a92-bd00-a390e18422eb
Adult-HIV-Project1_1 death_notification_management dd5930bc-4fc3-4ea3-8395-d6375c77023a
Adult-HIV-Project1_1 initial_death_notification 1fbd572e-9748-4b07-8bf0-739cbc6cc94a
Adult-HIV-Project1_1 eligibility_screening_form da3037a0-5334-4482-b4c1-d0e6f035a645
Adult-HIV-Project1_1 consent_tracking 4698e980-8860-4aea-9c8f-3e76d2e3c12e


#### 3.1 

In [5]:
import requests
import pandas as pd
import duckdb
# from ci_utils import connect_db

# pid=441 project 3.1 
data = {
    'token': '0BCBD93037F49F95EA6812C58F12502E',
    'content': 'record',
    'action': 'export',
    'format': 'json',
    'type': 'eav',
    'csvDelimiter': '',
    'rawOrLabel': 'raw',
    'rawOrLabelHeaders': 'raw',
    'exportCheckboxLabel': 'true', # ignored if 'type': 'eav'
    'exportSurveyFields': 'false',
    'exportDataAccessGroups': 'true',
    'returnFormat': 'json'
}
r = requests.post('https://champs.emory.edu/redcap/api/',data=data)
print(f'HTTP Status: {r.status_code}')
df = pd.DataFrame.from_dict(r.json())
df.to_csv('data/redcap_hiv_3_1.csv', index=False)

HTTP Status: 200


In [ ]:
import pandas as pd
import sqlalchemy as sa
from ci_utils import connect_db
CSV_FILE_NAME = 'data/redcap_hiv_3_1.csv'
STG_TABLE_NAME = 'HIVProject3_1_stg'
df = pd.read_csv(CSV_FILE_NAME, dtype=str, na_filter=False)
DATA_DICT_FILE_NAME = 'data/AdultHIVProject3_1_DataDict_2024-11-07.csv'
DATA_DICT_APPEND_FILE_NAME = 'data/AdultHIVProject3_1_DataDict_append.csv'


if not df.empty:
    engine = connect_db.conn_qa
    with engine.connect() as conn:
        #  use function create_data_dict_df to create data dictionary file
        data_dict_df = create_data_dict_df(DATA_DICT_FILE_NAME, DATA_DICT_APPEND_FILE_NAME)
        # rename df columns to match the staging table columns names
        df.columns = df.rename(columns= {'record': 'ChampsId','redcap_repeat_instrument': 'RepeatInstrument',
                                         'redcap_repeat_instance': 'RepeatInstance', 
                                         'field_name': 'FieldName', 'value': 'FieldValue'}).columns
        truncate_stg = sa.text(f"TRUNCATE TABLE stg.{STG_TABLE_NAME}")
        conn.execute(truncate_stg)
        df.to_sql(STG_TABLE_NAME, conn, schema='stg', if_exists='append', index=False)
        # data_dict_df.to_sql('HIVDataDictProj3_1', conn, schema='stg', if_exists='replace', index=False) if not data_dict_df.empty else None
        
        conn.commit()
df.head()        

### Add MITSProcedure record
```sql
insert into MITSProcedure (Id, ChampsId, MITSPerformed, SpecimenKitId, DateBodyReceived, 
MITSLocation, DateProcedureStarted, TimeProcedureStarted, TimeProcedureCompleted,
SexOfDeceased , FileName, 
Active , CreatedOn, ModifiedOn,UploadedOn)
select * from stg.vw_HIVMitsProcedure;